# TACO demo

This notebook demonstrates the four inference modes of the TACO model:

1. **Per-row scoring** — score each TCR against the peptide in its row
2. **Panel scoring** — score every TCR against a list of peptides (TCRs × peptides matrix)
3. **Beta-only scoring** — score using only TCR-β CDRs (alpha loops excluded from geometry)
4. **Beta-only panel** — combine panel mode with beta-only scoring


## Example data

TACO scores TCR–peptide pairs from CDR sequences alone.
The required columns are `cdr1a`, `cdr2a`, `cdr3a`, `cdr1b`, `cdr2b`, `cdr3b`,
and `peptide`, all as upper-case one-letter amino-acid strings with no gaps.

Below we create a small set of made-up TCRs and peptides for illustration.
The data cell also prepares `df_tcr` (unique TCRs) and `peptides` (a list of
peptide sequences), which are used in the panel-mode examples.

In [3]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "cdr1a":  ["DSAIYN", "TSGFNG", "VSGLRG", "DSAIYN", "TSGFNG"],
    "cdr2a":  ["IQSSQRE", "GLTSEGR", "LYSAGEE", "IQSSQRE", "GLTSEGR"],
    "cdr3a":  ["CAVRDNARLMF", "CAGGTSYGKLTF", "CAVNAGNMLTF", "CAVRDNARLMF", "CAGGTSYGKLTF"],
    "cdr1b":  ["MNHEY", "SGHNS", "SGHAT", "MNHEY", "SGHNS"],
    "cdr2b":  ["SVGAGI", "FVKESK", "FQNNGV", "SVGAGI", "FVKESK"],
    "cdr3b":  ["CASSFSTCSANYGYTF", "CASSYSDRGSYEQYF", "CASSLGQAYEQYF", "CASSFSTCSANYGYTF", "CASSYSDRGSYEQYF"],
    "peptide": ["GILGFVFTL", "GILGFVFTL", "GILGFVFTL", "NLVPMVATV", "NLVPMVATV"],
})
print(f"{len(df)} rows, {df['peptide'].nunique()} peptides")

cdr_cols = ["cdr1a", "cdr2a", "cdr3a", "cdr1b", "cdr2b", "cdr3b"]
df_tcr = df[cdr_cols].drop_duplicates().reset_index(drop=True)
peptides = ["GILGFVFTL", "NLVPMVATV", "GLCTLVAML"]
print(f"{len(df_tcr)} unique TCRs, {len(peptides)} peptides for panel mode")

df

5 rows, 2 peptides
3 unique TCRs, 3 peptides for panel mode


,cdr1a,cdr2a,cdr3a,cdr1b,cdr2b,cdr3b,peptide
0,DSAIYN,IQSSQRE,CAVRDNARLMF,MNHEY,SVGAGI,CASSFSTCSANYGYTF,GILGFVFTL
1,TSGFNG,GLTSEGR,CAGGTSYGKLTF,SGHNS,FVKESK,CASSYSDRGSYEQYF,GILGFVFTL
2,VSGLRG,LYSAGEE,CAVNAGNMLTF,SGHAT,FQNNGV,CASSLGQAYEQYF,GILGFVFTL
3,DSAIYN,IQSSQRE,CAVRDNARLMF,MNHEY,SVGAGI,CASSFSTCSANYGYTF,NLVPMVATV
4,TSGFNG,GLTSEGR,CAGGTSYGKLTF,SGHNS,FVKESK,CASSYSDRGSYEQYF,NLVPMVATV


## 1. Per-row scoring (DataFrame mode)

Each row has a `peptide` column — the model scores the TCR in that row against
that specific peptide. Returns one score per row.

`score()` returns log-odds; `predict_proba()` returns the corresponding
probability (sigmoid of the score).

In [4]:
from taco_model import TACOModel
TACO = TACOModel.load("data/taco/taco_model.json")

df["taco_score"] = TACO.score(df)
df["taco_prob"]  = TACO.predict_proba(df)
# df["taco_score"] = TACO.score(df, parallel=True) # For very large inferences (>1M), pass parallel=True
df[["cdr3b", "peptide", "taco_score", "taco_prob"]]

,cdr3b,peptide,taco_score,taco_prob
0,CASSFSTCSANYGYTF,GILGFVFTL,-4.315547,0.013183
1,CASSYSDRGSYEQYF,GILGFVFTL,-3.505517,0.029156
2,CASSLGQAYEQYF,GILGFVFTL,-3.490057,0.029596
3,CASSFSTCSANYGYTF,NLVPMVATV,-3.149018,0.041130
4,CASSYSDRGSYEQYF,NLVPMVATV,-4.093526,0.016407


**Validation:** with the example data above, TACO should produce these scores
(in row order):

| cdr3b | peptide | score | prob |
|---|---|---|---|
| CASSFSTCSANYGYTF | GILGFVFTL | −4.32 | 0.013 |
| CASSYSDRGSYEQYF | GILGFVFTL | −3.51 | 0.029 |
| CASSLGQAYEQYF | GILGFVFTL | −3.49 | 0.030 |
| CASSFSTCSANYGYTF | NLVPMVATV | −3.15 | 0.041 |
| CASSYSDRGSYEQYF | NLVPMVATV | −4.09 | 0.016 |

If your output matches (to two decimal places), the model is loaded correctly.

## 2. Panel scoring (TCRs × peptides)

Score every TCR against a list of peptides without building the full
cross-product DataFrame yourself. Pass a TCR-only DataFrame and a list of
peptides. Returns a DataFrame of shape (n_tcr, n_pep).

This is more efficient than building the cross product because the TCR geometry
is computed once and reused for every peptide.

In [5]:
from taco_model import TACOModel
TACO = TACOModel.load("data/taco/taco_model.json")

panel = TACO.score(df_tcr, peptides=peptides)
panel_prob = TACO.predict_proba(df_tcr, peptides=peptides)
# panel = TACO.score(df_tcr, peptides=peptides, parallel=True) # For very large inferences (>1M), pass parallel=True
print(f"Panel shape: {panel.shape}  (TCRs × peptides)")
print("Log-odds:")
display(panel)
print("Probabilities:")
display(panel_prob)

Panel shape: (3, 3)  (TCRs × peptides)
Log-odds:


,GILGFVFTL,NLVPMVATV,GLCTLVAML
0,-4.315547,-3.149018,-2.845747
1,-3.505517,-4.093526,-4.397321
2,-3.490057,-4.246110,-3.988603


Probabilities:


,GILGFVFTL,NLVPMVATV,GLCTLVAML
0,0.013183,0.041130,0.054902
1,0.029156,0.016407,0.012161
2,0.029596,0.014118,0.018189


## 3. Beta-only scoring

Score using only TCR-β CDRs — alpha loop contributions are excluded entirely
from the contact geometry.

Beta-only scores are useful for quantifying the model's alpha chain's contribution (should not be interpreted as biological ground truth) and
for datasets where only TCR-β is available.


In [6]:
from taco_model import TACOModel
TACO = TACOModel.load("data/taco/taco_model.json")

df["taco_beta"] = TACO.score_beta(df)
# df["taco_beta"] = TACO.score_beta(df, parallel=True) # For very large inferences (>1M), pass the parallel=True argument to enable parallel processing
df[["cdr3b", "peptide", "taco_score", "taco_beta"]]


,cdr3b,peptide,taco_score,taco_beta
0,CASSFSTCSANYGYTF,GILGFVFTL,-4.315547,-3.357467
1,CASSYSDRGSYEQYF,GILGFVFTL,-3.505517,-3.452592
2,CASSLGQAYEQYF,GILGFVFTL,-3.490057,-3.399531
3,CASSFSTCSANYGYTF,NLVPMVATV,-3.149018,-3.689266
4,CASSYSDRGSYEQYF,NLVPMVATV,-4.093526,-4.492371


In [7]:
from taco_model import TACOModel
TACO = TACOModel.load("data/taco/taco_model.json")

panel_beta = TACO.score_beta(df_tcr, peptides=peptides)
print(f"Panel shape: {panel_beta.shape}  (TCRs × peptides)")
panel_beta


Panel shape: (3, 3)  (TCRs × peptides)


,GILGFVFTL,NLVPMVATV,GLCTLVAML
0,-3.357467,-3.689266,-2.635111
1,-3.452592,-4.492371,-5.147755
2,-3.399531,-4.115222,-3.863722


## Interpreting scores

- **`score()`** returns transferable log-odds with a single global intercept.
  Scores are directly comparable across antigens — no normalization needed.
  Higher = stronger predicted activation.

- **`predict_proba()`** returns the activation probability — the sigmoid of the
  score. Same interface as `score()`. Values in [0, 1].

- **`score_beta()`** returns log-odds using only TCR-β loop contributions, with
  the same global bias. Scores are directly comparable across antigens, same as
  `score()`. Useful when only TCR-β is available or for quantifying the alpha
  chain's contribution.
